<a href="https://colab.research.google.com/github/belkacemkadri29-png/Power-System-Datasets-Under-Advanced-Cyber-Attacks-scenarios/blob/main/Simulation%20of%20poisoned%20attack%20scenario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import torch.nn as nn
import pandas as pd


input_file = "/content/57-training_set.xlsx"
poison_percent = 0.01
epsilon = 0.3
alpha = 0.01
num_iter = 10


df = pd.read_excel(input_file)
column_names = df.columns.tolist()
has_labels = 'label' in df.columns

if has_labels:
    feature_columns = df.drop(columns=['label']).columns.tolist()
    features = df[feature_columns].values
    labels = df['label'].values
else:
    feature_columns = df.columns.tolist()
    features = df.values
    labels = None

X_clean = torch.tensor(features, dtype=torch.float)
if has_labels:
    y_clean = torch.tensor(labels, dtype=torch.long)

num_samples, num_features = X_clean.shape

class SimpleMLP(nn.Module):
    def __init__(self, input_dim):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 2 if has_labels else input_dim)
        )
    def forward(self, x):
        return self.net(x)

model = SimpleMLP(num_features)
criterion = nn.CrossEntropyLoss() if has_labels else nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


for epoch in range(10):
    optimizer.zero_grad()
    output = model(X_clean)
    target = y_clean if has_labels else X_clean
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()


X_poisoned = X_clean.clone().detach()
num_poison = int(poison_percent * num_samples)
poison_indices = torch.randperm(num_samples)[:num_poison]

for _ in range(num_iter):
    x_batch = X_poisoned[poison_indices].clone().detach().requires_grad_(True)
    target = y_clean[poison_indices] if has_labels else X_clean[poison_indices]

    output = model(x_batch)
    loss = criterion(output, target)
    model.zero_grad()
    loss.backward()

    grad = x_batch.grad
    perturbed = x_batch + alpha * grad.sign()
    perturbed = torch.max(torch.min(perturbed, X_clean[poison_indices] + epsilon),
                          X_clean[poison_indices] - epsilon)

    X_poisoned[poison_indices] = perturbed.detach()


df_poisoned = pd.DataFrame(X_poisoned.numpy(), columns=feature_columns)
if has_labels:
    df_poisoned['label'] = labels


output_file = f"poisoned_dataset_PGD_{int(poison_percent*100)}pct.xlsx"
df_poisoned.to_excel("/results/poisoned_dataset_PGD_{int(poison_percent*100)}pct.xlsx", index=False)

print(f"Données empoisonnées ({poison_percent*100}%) sauvegardées dans '{output_file}'")
